# ChartCheck Spec Extraction & Quality Verification
Runs the 3B SFT LoRA model on ChartCheck images, extracts specs,
and verifies quality for downstream MACE claim verification training.

In [ ]:
!pip install unsloth qwen-vl-utils datasets tqdm

In [ ]:
# ==========================================
# 1. CONFIGURATION
# Update these paths to match your workspace
# ==========================================
import os

os.environ["HF_HOME"] = "/workspace/huggingface_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"

# --- Paths ---
# Your 3B SFT checkpoint (the one we verified produces clean JSON)
SFT_CHECKPOINT = "/workspace/ImageToSpec_Stage1/qwen3b_lora_sft_bf16_final/qwen3b_lora_sft_bf16_final"

# ChartCheck dataset location
# Download from: https://github.com/doughendry/chartcheck
# Expected format: CSV or JSON with columns: image_path, claim, label (Supported/Refuted)
CHARTCHECK_DIR   = "/workspace/chartcheck"              # root folder
CHARTCHECK_CSV   = os.path.join(CHARTCHECK_DIR, "chartcheck.csv")  # adjust filename if needed
CHARTCHECK_IMGS  = os.path.join(CHARTCHECK_DIR, "images")          # image folder

# Output
OUTPUT_FILE  = "/workspace/chartcheck_with_specs.json"
CACHE_FILE   = "/workspace/chartcheck_spec_cache.json"
REPORT_FILE  = "/workspace/chartcheck_quality_report.json"

# Generation config
MAX_NEW_TOKENS = 3072
MAX_PIXELS     = 768 * 768

print("Config loaded.")

In [ ]:
# ==========================================
# 2. LOAD MODEL
# Uses Unsloth — same as your GRPO training setup
# ==========================================
import torch
from unsloth import FastVisionModel

print(f"Loading model from {SFT_CHECKPOINT}...")
model, processor = FastVisionModel.from_pretrained(
    SFT_CHECKPOINT,
    load_in_4bit=False,
)
FastVisionModel.for_inference(model)
model.eval()
print("✅ Model loaded.")
print(f"   Device: {next(model.parameters()).device}")
print(f"   Dtype:  {next(model.parameters()).dtype}")

In [ ]:
# ==========================================
# 3. HELPERS
# extract_json: the robust brace-counting parser
# that fixes the trailing-garbage problem found
# during GRPO training.
# ==========================================
import re
import json

def extract_json(text):
    """
    Extracts the first complete, balanced JSON object from model output.
    Handles trailing garbage appended after the closing brace
    e.g. '}}],"chart_type":"scatter"}' appended after the root closes.
    """
    text = text.strip()

    # Try markdown block first
    json_match = re.search(r'```json\n(.*?)\n```', text, re.DOTALL)
    if json_match:
        text = json_match.group(1)

    start = text.find('{')
    if start == -1:
        return None

    depth = 0
    in_string = False
    escape_next = False

    for i, ch in enumerate(text[start:], start):
        if escape_next:
            escape_next = False
            continue
        if ch == '\\' and in_string:
            escape_next = True
            continue
        if ch == '"' and not escape_next:
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except json.JSONDecodeError:
                    return None
    return None


def safe_convert_to_rgb(img_path):
    """
    Converts any image to RGB, handling transparency by pasting
    onto a white background (prevents text going black on dark backgrounds).
    """
    from PIL import Image
    img = Image.open(img_path)
    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
        img = img.convert('RGBA')
        bg = Image.new('RGB', img.size, (255, 255, 255))
        bg.paste(img, mask=img)
        return bg
    return img.convert('RGB')


print("Helpers defined.")

In [ ]:
# ==========================================
# 4. INFERENCE FUNCTION
# Uses the same prompt format as GRPO training
# so the model sees a familiar instruction.
# Greedy decoding (do_sample=False) for
# deterministic, stable extraction.
# ==========================================

SYSTEM_PROMPT = (
    "You are a chart analysis assistant. "
    "Extract the chart specification as a single valid JSON object. "
    "Output only the JSON with no markdown, no explanation, no extra text."
)

USER_PROMPT = (
    "Analyze this chart image and extract its complete specification as JSON. "
    "Include: title, chart_type, panel_count, panel_layout, panels (with topo, axes, ser). "
    "For each series include name, color, data points. "
    "Output only valid JSON."
)


def run_inference(image):
    """
    Runs the 3B SFT model on a PIL image.
    Returns parsed spec dict or None if extraction fails.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": USER_PROMPT},
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt",
        padding=True,
    ).to("cuda")

    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,   # greedy — deterministic for quality check
        )

    generated_ids = out[0][prompt_len:]
    decoded = processor.decode(generated_ids, skip_special_tokens=True)
    return extract_json(decoded), decoded


print("Inference function defined.")

In [ ]:
# ==========================================
# 5. LOAD CHARTCHECK DATASET
#
# ChartCheck format (adjust field names below
# if your CSV/JSON uses different column names):
#   image: filename or path to chart image
#   claim: natural language claim about the chart
#   label: 'Supported' or 'Refuted'
# ==========================================
import pandas as pd

# --- Load ---
# Adjust to .json if your ChartCheck is JSON format:
df = pd.read_csv(CHARTCHECK_CSV)
print(f"ChartCheck loaded: {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print(df.head(3))

In [ ]:
# ==========================================
# 5b. FIELD MAPPING
# Rename columns to match our internal schema.
# Update these if your ChartCheck has different names.
# ==========================================

# --- Adjust these to match your actual column names ---
FIELD_IMAGE  = "image"      # column with image filename/path
FIELD_CLAIM  = "claim"      # column with claim text
FIELD_LABEL  = "label"      # column with Supported/Refuted

# Normalize
df = df.rename(columns={
    FIELD_IMAGE: "image_file",
    FIELD_CLAIM: "claim",
    FIELD_LABEL: "label"
})

# Build full image paths
df["image_path"] = df["image_file"].apply(
    lambda f: os.path.join(CHARTCHECK_IMGS, os.path.basename(str(f)))
)

# Check how many images exist on disk
df["image_exists"] = df["image_path"].apply(os.path.exists)
print(f"Images found on disk: {df['image_exists'].sum()} / {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")

# Filter to only rows with existing images
df = df[df["image_exists"]].reset_index(drop=True)
print(f"\nProceeding with {len(df)} rows that have images.")

In [ ]:
# ==========================================
# 6. QUICK SANITY CHECK — 3 SAMPLES
# Run inference on 3 samples before the full
# loop to catch any issues early.
# ==========================================
from PIL import Image

print("=" * 60)
print("SANITY CHECK: 3 samples")
print("=" * 60)

for i in range(min(3, len(df))):
    row = df.iloc[i]
    image = safe_convert_to_rgb(row["image_path"])
    spec, raw_text = run_inference(image)

    print(f"\n--- Sample {i+1} ---")
    print(f"Image : {row['image_path']}")
    print(f"Claim : {row['claim']}")
    print(f"Label : {row['label']}")
    print(f"Parse : {'✅ Valid JSON' if spec else '❌ FAILED'}")
    if spec:
        print(f"Type  : {spec.get('chart_type', 'unknown')}")
        print(f"Panels: {spec.get('panel_count', '?')}")
        n_series = sum(len(p.get('ser', [])) for p in spec.get('panels', []))
        print(f"Series: {n_series} total")
        # Show first 200 chars of spec
        print(f"Spec preview: {json.dumps(spec)[:200]}...")
    else:
        print(f"Raw output tail: {raw_text[-150:]}")

In [ ]:
# ==========================================
# 7. FULL EXTRACTION LOOP
# - Caches results per image path
# - Resumes from partial runs
# - Saves after every sample
# ==========================================
from tqdm import tqdm

# Load or initialize cache
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'r') as f:
        spec_cache = json.load(f)
    print(f"Loaded cache: {len(spec_cache)} entries")
else:
    spec_cache = {}
    print("Starting fresh cache.")

# Load or initialize output
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        results = json.load(f)
    print(f"Resuming: {len(results)} rows already done.")
else:
    results = []

start_idx = len(results)
remaining = df.iloc[start_idx:]

if len(remaining) == 0:
    print("✅ Already complete!")
else:
    print(f"Processing {len(remaining)} remaining rows (starting at index {start_idx})...")

    for _, row in tqdm(remaining.iterrows(), total=len(remaining)):
        img_path = row["image_path"]
        result = {
            "image_path": img_path,
            "claim": row["claim"],
            "label": row["label"],
            "spec": None,
            "parse_success": False,
            "raw_tail": ""
        }

        try:
            # Use cache if available
            if img_path in spec_cache:
                spec = spec_cache[img_path]
                result["spec"] = spec
                result["parse_success"] = spec is not None
            else:
                image = safe_convert_to_rgb(img_path)
                spec, raw_text = run_inference(image)
                result["spec"] = spec
                result["parse_success"] = spec is not None
                result["raw_tail"] = raw_text[-100:] if not spec else ""

                # Update cache
                spec_cache[img_path] = spec
                with open(CACHE_FILE, 'w') as f:
                    json.dump(spec_cache, f)

        except Exception as e:
            result["error"] = str(e)
            print(f"\nError on {img_path}: {e}")

        results.append(result)

        # Save after every sample
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(results, f, indent=2)

    print(f"\n✅ Done. {len(results)} total rows saved to {OUTPUT_FILE}")

In [ ]:
# ==========================================
# 8. QUALITY REPORT
# Measures parse rate, chart type distribution,
# series coverage, and per-label breakdown.
# This is your spec quality benchmark.
# ==========================================
from collections import Counter, defaultdict

total = len(results)
parsed = [r for r in results if r["parse_success"] and r["spec"]]
failed = [r for r in results if not r["parse_success"]]

parse_rate = len(parsed) / total * 100

# Chart type distribution
chart_types = Counter(r["spec"].get("chart_type", "unknown") for r in parsed)

# Series coverage
series_counts = []
empty_series = 0
for r in parsed:
    spec = r["spec"]
    n = sum(len(p.get("ser", [])) for p in spec.get("panels", []))
    series_counts.append(n)
    if n == 0:
        empty_series += 1

# Per-label breakdown
label_parse = defaultdict(lambda: {"total": 0, "parsed": 0})
for r in results:
    label_parse[r["label"]]["total"] += 1
    if r["parse_success"]:
        label_parse[r["label"]]["parsed"] += 1

# Panel count distribution
panel_counts = Counter(r["spec"].get("panel_count", 0) for r in parsed)

# --- Print report ---
print("=" * 60)
print("SPEC QUALITY REPORT")
print("=" * 60)
print(f"Total samples    : {total}")
print(f"Parse success    : {len(parsed)} ({parse_rate:.1f}%)")
print(f"Parse failed     : {len(failed)} ({100-parse_rate:.1f}%)")
print(f"Empty series     : {empty_series} of {len(parsed)} parsed")
print()
print(f"Avg series/chart : {sum(series_counts)/max(len(series_counts),1):.1f}")
print(f"Min series       : {min(series_counts) if series_counts else 0}")
print(f"Max series       : {max(series_counts) if series_counts else 0}")
print()
print("Chart types:")
for ct, count in chart_types.most_common():
    print(f"  {ct:20s}: {count}")
print()
print("Panel counts:")
for pc, count in sorted(panel_counts.items()):
    print(f"  {pc} panel(s): {count}")
print()
print("Per-label parse rate:")
for label, stats in label_parse.items():
    rate = stats["parsed"] / stats["total"] * 100
    print(f"  {label:12s}: {stats['parsed']}/{stats['total']} ({rate:.1f}%)")

# Save report
report = {
    "total": total,
    "parsed": len(parsed),
    "failed": len(failed),
    "parse_rate_pct": round(parse_rate, 2),
    "empty_series_count": empty_series,
    "avg_series_per_chart": round(sum(series_counts)/max(len(series_counts),1), 2),
    "chart_type_distribution": dict(chart_types),
    "panel_count_distribution": {str(k): v for k, v in panel_counts.items()},
    "per_label_parse_rate": {
        k: {"parsed": v["parsed"], "total": v["total"],
            "rate_pct": round(v["parsed"]/v["total"]*100, 1)}
        for k, v in label_parse.items()
    }
}
with open(REPORT_FILE, 'w') as f:
    json.dump(report, f, indent=2)
print(f"\nReport saved to {REPORT_FILE}")

In [ ]:
# ==========================================
# 9. MANUAL INSPECTION — 10 RANDOM SAMPLES
# Visual check: does the spec content actually
# match what the claim is talking about?
# This is the key quality gate before using
# these specs for MACE training.
# ==========================================
import random
from PIL import Image
import IPython.display as ipd

random.seed(42)
sample_pool = [r for r in results if r["parse_success"] and r["spec"]]
inspection_samples = random.sample(sample_pool, min(10, len(sample_pool)))

print("=" * 60)
print("MANUAL INSPECTION SAMPLES")
print("Check: does the spec data support or refute the claim?")
print("=" * 60)

for i, r in enumerate(inspection_samples):
    print(f"\n{'='*50}")
    print(f"Sample {i+1}")
    print(f"Claim : {r['claim']}")
    print(f"Label : {r['label']}")

    spec = r["spec"]
    print(f"Chart : {spec.get('chart_type', '?')} | Panels: {spec.get('panel_count', '?')}")

    for pi, panel in enumerate(spec.get("panels", [])):
        print(f"  Panel {pi}: {panel.get('topo', {}).get('type', '?')} / {panel.get('topo', {}).get('sub', '')}")
        for si, ser in enumerate(panel.get("ser", [])):
            data_preview = str(ser.get("data", []))[:120]
            print(f"    Series '{ser.get('name', '?')}': {data_preview}")

    # Show image inline if running in Jupyter
    try:
        img = Image.open(r["image_path"])
        img.thumbnail((400, 300))
        ipd.display(img)
    except Exception:
        print(f"  [Image: {r['image_path']}]")

In [ ]:
# ==========================================
# 10. FAILURE ANALYSIS
# Inspect what went wrong on failed parses.
# ==========================================
print("=" * 60)
print(f"FAILURE ANALYSIS — {len(failed)} failed samples")
print("=" * 60)

for i, r in enumerate(failed[:10]):  # show first 10
    print(f"\nFail {i+1}: {r['image_path']}")
    print(f"  Claim: {r['claim'][:80]}")
    print(f"  Label: {r['label']}")
    print(f"  Raw tail: {r.get('raw_tail', 'N/A')}")

# Categorize failure modes
truncated = [r for r in failed if r.get("raw_tail", "").count('}') > 0]
empty_out = [r for r in failed if not r.get("raw_tail", "").strip()]
other      = [r for r in failed if r not in truncated and r not in empty_out]

print(f"\nFailure categories:")
print(f"  Truncated mid-JSON (trailing braces): {len(truncated)}")
print(f"  Empty output                        : {len(empty_out)}")
print(f"  Other                               : {len(other)}")

In [ ]:
# ==========================================
# 11. BUILD MACE-READY DATASET
# Converts extracted specs into the unified
# MACE training format:
# {claim, evidence_type, evidence, label}
# Only includes successfully parsed specs.
# ==========================================

MACE_OUTPUT = "/workspace/chartcheck_mace_pairs.json"

mace_pairs = []
skipped = 0

for r in results:
    if not r["parse_success"] or not r["spec"]:
        skipped += 1
        continue

    spec = r["spec"]

    # Quality gate: skip specs with no series data
    has_data = any(
        len(s.get("data", [])) > 0
        for p in spec.get("panels", [])
        for s in p.get("ser", [])
    )
    if not has_data:
        skipped += 1
        continue

    mace_pairs.append({
        "claim": r["claim"],
        "evidence_type": "chart",
        "evidence": json.dumps(spec),   # spec JSON as string for the verifier
        "label": r["label"],            # Supported / Refuted
        "image_path": r["image_path"],  # keep for debugging
    })

with open(MACE_OUTPUT, 'w') as f:
    json.dump(mace_pairs, f, indent=2)

print(f"MACE pairs built: {len(mace_pairs)}")
print(f"Skipped          : {skipped}")
print(f"Saved to         : {MACE_OUTPUT}")
print()
print("Label distribution in MACE pairs:")
label_counts = Counter(r["label"] for r in mace_pairs)
for label, count in label_counts.items():
    print(f"  {label}: {count}")

In [ ]:
# ==========================================
# 12. DECISION GATE
# Based on the quality report, tells you
# whether the specs are ready for MACE training
# or need fixes first.
# ==========================================

print("=" * 60)
print("DECISION GATE")
print("=" * 60)

issues = []

if parse_rate < 80:
    issues.append(f"❌ Parse rate {parse_rate:.1f}% is below 80% threshold")
    issues.append("   → Check if model checkpoint path is correct")
    issues.append("   → Run failure analysis above to identify pattern")
else:
    print(f"✅ Parse rate {parse_rate:.1f}% — acceptable")

empty_rate = empty_series / max(len(parsed), 1) * 100
if empty_rate > 10:
    issues.append(f"❌ {empty_rate:.1f}% of parsed specs have no series data")
    issues.append("   → Model is producing structural specs without data")
    issues.append("   → These are filtered out in step 11 automatically")
else:
    print(f"✅ Empty series rate {empty_rate:.1f}% — acceptable")

if len(mace_pairs) < 500:
    issues.append(f"❌ Only {len(mace_pairs)} usable pairs — may be too few for training")
    issues.append("   → Consider augmenting with synthetic claims (see plan)")
else:
    print(f"✅ {len(mace_pairs)} usable MACE pairs — sufficient for Stage 1 training")

if issues:
    print()
    print("Issues found:")
    for issue in issues:
        print(issue)
else:
    print()
    print("✅ All checks passed. Specs are ready for MACE Stage 1 training.")
    print(f"   Next step: combine {MACE_OUTPUT} with InfoTabs and SciFact")
    print("   and run Stage 1 SFT.")